In [1]:
import os
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt

# ----------------------------
# Paths (edit 'root' if needed)
# ----------------------------
root = r"C:\Drought"
paths = {
    "VDI_dir": os.path.join(root, r"Regridding and data clipping\Final Analysis\VDI"),
    "PDSI":    os.path.join(root, r"Regridding and data clipping\PDSI", "TerraClimate_PDSI_0p05deg_INDIA.nc"),
    "VHI":     os.path.join(root, r"Regridding and data clipping\VHI", "MODIS_VCI_TCI_VHI_India_0p05deg_2001-2023.nc"),
    "SPI":     os.path.join(root, r"SPI", "SPI_CHIRPS_k_1_3_6_9_12_2000_2023_0p05deg.nc"),
    "GRACE":   os.path.join(root, r"Regridding and data clipping\GRACE", "GRACE_JPL_mascon_India_0p05deg_2003-2023.nc"),
    "EDDI":    os.path.join(root, r"EDDI", "EDDI_005deg_200012-202312.nc"),
    "LULC_dir":os.path.join(root, r"Land Cover"),
    "out":     os.path.join(root, r"Maps_out"),
}
os.makedirs(paths["out"], exist_ok=True)

# ----------------------------
# Helpers
# ----------------------------
def open_nc(fp):
    """Open NetCDF and standardize to lat/lon (ascending)."""
    ds = xr.open_dataset(fp)
    rename = {}
    if "latitude" in ds: rename["latitude"] = "lat"
    if "longitude" in ds: rename["longitude"] = "lon"
    if "y" in ds and "lat" not in ds: rename["y"] = "lat"
    if "x" in ds and "lon" not in ds: rename["x"] = "lon"
    if rename: ds = ds.rename(rename)
    if "lat" in ds.coords: ds = ds.sortby("lat")
    if "lon" in ds.coords: ds = ds.sortby("lon")
    return ds

def first_var(ds, candidates):
    """Pick first present variable from candidates (case-insensitive); else first time/lat/lon var."""
    cl = [c.lower() for c in candidates]
    for v in ds.data_vars:
        if v.lower() in cl: return v
    for v in candidates:
        if v in ds.data_vars: return v
    for v in ds.data_vars:
        dims = ds[v].dims
        if "time" in dims and (("lat" in dims) or ("y" in dims)) and (("lon" in dims) or ("x" in dims)):
            return v
    raise ValueError(f"Could not find expected variable among: {list(ds.data_vars)}")

def ensure_datetime_monthly(da):
    """
    Ensure 'time' is datetime64 and monthly.
    Uses resample to month-start ('MS') means; robust across xarray/pandas versions.
    """
    if "time" not in da.dims:
        raise ValueError("DataArray must have 'time' dimension.")
    if not np.issubdtype(da["time"].dtype, np.datetime64):
        da = da.assign_coords(time=pd.to_datetime(da["time"].values))
    da = da.sortby("time")
    if not np.issubdtype(da.dtype, np.number):
        da = da.astype("float64")
    da_m = da.resample(time="MS").mean(skipna=True)
    # Drop months that are all-NaN across space (optional cleanliness)
    if "time" in da_m.coords:
        all_nan = ~np.isfinite(da_m).any(dim=[d for d in da_m.dims if d != "time"])
        if all_nan.any():
            keep = (~all_nan).values
            if keep.any():
                da_m = da_m.sel(time=da_m["time"].values[keep])
    return da_m

def interp_to_ref_grid(da, ref_lat, ref_lon, method="nearest"):
    """Interpolate da to the reference (lat,lon) grid."""
    if "lat" not in da.coords and "y" in da.coords: da = da.rename({"y":"lat"})
    if "lon" not in da.coords and "x" in da.coords: da = da.rename({"x":"lon"})
    if "lat" in da.coords: da = da.sortby("lat")
    if "lon" in da.coords: da = da.sortby("lon")
    same_lat = ("lat" in da.coords) and np.array_equal(da["lat"].values, ref_lat.values)
    same_lon = ("lon" in da.coords) and np.array_equal(da["lon"].values, ref_lon.values)
    if same_lat and same_lon:
        return da
    return da.interp(lat=ref_lat, lon=ref_lon, method=method)

def apply_mask(da, mask, thr=0.5, interp_method="nearest"):
    """Apply biome mask (fraction >= thr). Interpolate mask to da grid if needed."""
    if "lat" in da.coords: da = da.sortby("lat")
    if "lon" in da.coords: da = da.sortby("lon")
    if "lat" in mask.coords: mask = mask.sortby("lat")
    if "lon" in mask.coords: mask = mask.sortby("lon")
    if "time" in mask.dims: mask = mask.mean("time")
    need_interp = (
        mask.sizes.get("lat", -1) != da.sizes.get("lat", -1) or
        mask.sizes.get("lon", -1) != da.sizes.get("lon", -1) or
        ("lat" in mask.coords and "lat" in da.coords and not np.array_equal(mask["lat"].values, da["lat"].values)) or
        ("lon" in mask.coords and "lon" in da.coords and not np.array_equal(mask["lon"].values, da["lon"].values))
    )
    if need_interp:
        mask = mask.interp(lat=da["lat"], lon=da["lon"], method=interp_method)
    mask_bool = xr.where(mask >= thr, 1.0, np.nan)
    return da * mask_bool

def area_weighted_biome_mean(da_masked):
    """Area-weighted (cos lat) mean over lat,lon for a masked DataArray."""
    lat = da_masked["lat"]
    w_lat = np.cos(np.deg2rad(lat))
    W, _ = xr.broadcast(w_lat, da_masked["lon"])
    # Use first available month as stencil to zero out weights outside biome
    stencil = da_masked.isel(time=0, drop=True).notnull()
    W = W.where(stencil, 0.0)
    return da_masked.weighted(W).mean(("lat", "lon"), skipna=True)

def zscore(da1d):
    """Standardize a 1D monthly series (mean 0, std 1)."""
    mu = da1d.mean("time", skipna=True)
    sd = da1d.std("time", skipna=True)
    return (da1d - mu) / sd

def intersect_many(series_list):
    """Return list of series restricted to the common time intersection (sorted)."""
    idx = pd.Index(series_list[0]["time"].values)
    for s in series_list[1:]:
        idx = idx.intersection(pd.Index(s["time"].values))
    idx = np.array(sorted(idx))
    return [s.sel(time=idx) for s in series_list], idx

# ----------------------------
# Load biome masks
# ----------------------------
mask_files = {
    "cropland":    os.path.join(paths["LULC_dir"], "lulc_cropland_frac_0p05deg.nc"),
    "forest":      os.path.join(paths["LULC_dir"], "lulc_forest_frac_0p05deg.nc"),
    "shrub_grass": os.path.join(paths["LULC_dir"], "lulc_shrub_grass_frac_0p05deg.nc"),
}
biome_masks = {}
for name, fp in mask_files.items():
    ds = open_nc(fp)
    v = first_var(ds, ["fraction","frac","cropland","forest","shrub_grass","lulc"])
    m = ds[v]
    if "y" in m.dims and "lat" not in m.dims: m = m.rename({"y":"lat"})
    if "x" in m.dims and "lon" not in m.dims: m = m.rename({"x":"lon"})
    if "time" in m.dims: m = m.mean("time")
    biome_masks[name] = m

# ----------------------------
# Load datasets
# ----------------------------
print("Loading datasets...")
# VDI: pick a plausible file in the folder
vdi_candidates = [f for f in os.listdir(paths["VDI_dir"]) if f.lower().endswith(".nc")]
prefer = [f for f in vdi_candidates if "vdi" in f.lower() and ("all" in f.lower() or "india" in f.lower())]
vdi_fp = os.path.join(paths["VDI_dir"], prefer[0] if prefer else vdi_candidates[0])
vdi_ds = open_nc(vdi_fp); vdi = vdi_ds[first_var(vdi_ds, ["VDI","vdi"])]

pdsi_ds = open_nc(paths["PDSI"]); pdsi = pdsi_ds[first_var(pdsi_ds, ["PDSI","pdsi"])]
vhi_ds  = open_nc(paths["VHI"]);  vhi  = vhi_ds[first_var(vhi_ds,  ["VHI","vhi"])]
spi_ds  = open_nc(paths["SPI"])
spi3    = spi_ds[first_var(spi_ds, ["SPI3","SPI_3","spi3","spi_3"])]
spi6    = spi_ds[first_var(spi_ds, ["SPI6","SPI_6","spi6","spi_6"])]
grace_ds= open_nc(paths["GRACE"]); grace = grace_ds[first_var(grace_ds, ["TWS","GAB_msc_Lmax180","tws_anom","TWSA","grace","mascon"])]
eddi_ds = open_nc(paths["EDDI"])
eddi3   = eddi_ds[first_var(eddi_ds, ["EDDI_03","EDDI3","eddi_03","EDDI_3","eddi3"])]
eddi6   = eddi_ds[first_var(eddi_ds, ["EDDI_06","EDDI6","eddi_06","EDDI_6","eddi6"])]

# Coerce numeric dtype where needed
for key, arr in [("vdi", vdi), ("pdsi", pdsi), ("vhi", vhi),
                 ("spi3", spi3), ("spi6", spi6), ("grace", grace),
                 ("eddi3", eddi3), ("eddi6", eddi6)]:
    if not np.issubdtype(arr.dtype, np.number):
        locals()[key] = arr.astype("float64")

# ----------------------------
# Prepare monthly and align to VDI grid (for consistent pixels)
# ----------------------------
print("Preparing monthly series & grids...")
vdi_m   = ensure_datetime_monthly(vdi)
ref_lat, ref_lon = vdi_m["lat"], vdi_m["lon"]

pdsi_m  = interp_to_ref_grid(ensure_datetime_monthly(pdsi),  ref_lat, ref_lon)
vhi_m   = interp_to_ref_grid(ensure_datetime_monthly(vhi),   ref_lat, ref_lon)
spi3_m  = interp_to_ref_grid(ensure_datetime_monthly(spi3),  ref_lat, ref_lon)
spi6_m  = interp_to_ref_grid(ensure_datetime_monthly(spi6),  ref_lat, ref_lon)
grace_m = interp_to_ref_grid(ensure_datetime_monthly(grace), ref_lat, ref_lon)
eddi3_m = interp_to_ref_grid(ensure_datetime_monthly(eddi3), ref_lat, ref_lon)
eddi6_m = interp_to_ref_grid(ensure_datetime_monthly(eddi6), ref_lat, ref_lon)

# ----------------------------
# Build biome-specific time series and plot
# ----------------------------
series_sources = [
    ("VDI",   vdi_m),
    ("PDSI",  pdsi_m),
    ("VHI",   vhi_m),
    ("SPI-3", spi3_m),
    ("SPI-6", spi6_m),
    ("GRACE", grace_m),
    ("EDDI-3", eddi3_m),
    ("EDDI-6", eddi6_m),
]

print("Computing biome-averaged series and plotting...")
for bname, bmask in biome_masks.items():
    # Mask and area-average for each variable
    ts_raw = []
    labels = []
    for label, da in series_sources:
        da_b = apply_mask(da, bmask, thr=0.5)      # mask biome (handles grid diffs)
        ts   = area_weighted_biome_mean(da_b)      # monthly area-weighted mean
        ts_raw.append(ts)
        labels.append(label)

    # Align time across all variables (common overlap)
    ts_aligned, common_time = intersect_many(ts_raw)

    # Standardize each series over common overlap
    ts_std = [zscore(s) for s in ts_aligned]

    # Plot
    plt.figure(figsize=(12, 5), facecolor="white")
    ax = plt.gca()
    for s, lab in zip(ts_std, labels):
        ax.plot(pd.to_datetime(s["time"].values), s.values, label=lab, linewidth=1.5)

    ax.axhline(0, color="k", linewidth=0.8, alpha=0.5)
    ax.set_title(f"{bname.capitalize()} – monthly standardized time series (area-weighted mean)", fontsize=13)
    ax.set_xlabel("Time"); ax.set_ylabel("Standardized value (z-score)")
    ax.grid(True, alpha=0.25)
    ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), ncol=1, frameon=False)

    out_png = os.path.join(paths["out"], f"Timeseries_allvars_{bname}.png")
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"Saved: {out_png}")

print("All biome time-series figures are ready.")


<>:2: SyntaxWarning: invalid escape sequence '\D'
<>:2: SyntaxWarning: invalid escape sequence '\D'
C:\Users\Vikas.Patel\AppData\Local\Temp\ipykernel_16620\3945849366.py:2: SyntaxWarning: invalid escape sequence '\D'
  """
C:\Users\Vikas.Patel\AppData\Local\anaconda3\Lib\site-packages\paramiko\pkey.py:82: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "cipher": algorithms.TripleDES,
C:\Users\Vikas.Patel\AppData\Local\anaconda3\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.Blowfish and will be removed from this module in 45.0.0.
  "class": algorithms.Blowfish,
C:\Users\Vikas.Patel\AppData\Local\anaconda3\Lib\site-packages\paramiko\transport.py:243: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES

Loading datasets...
Preparing monthly series & grids...
Computing biome-averaged series and plotting...
Saved: C:\Drought\Maps_out\Timeseries_allvars_cropland.png
Saved: C:\Drought\Maps_out\Timeseries_allvars_forest.png
Saved: C:\Drought\Maps_out\Timeseries_allvars_shrub_grass.png
All biome time-series figures are ready.


In [3]:
from matplotlib.colors import TwoSlopeNorm
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def to_drought_positive(da, kind):
    """
    Convert to 'higher = drier' convention.
    kind ∈ {'vdi','pdsi','spi','vhi','grace','eddi'}
    """
    if kind == "vdi":
        return da
    if kind == "pdsi":
        return -da
    if kind == "spi":
        return -da
    if kind == "grace":
        return -da
    if kind == "eddi":
        return da
    if kind == "vhi":
        vmin = float(da.min(skipna=True))
        vmax = float(da.max(skipna=True))
        if 0.0 <= vmin <= 100.0 and 0.0 <= vmax <= 100.0:
            return 100.0 - da
        else:
            return -da
    raise ValueError(f"Unknown kind: {kind}")

# Prepare drought-positive grids (reuse the already-monthly, VDI-aligned arrays)
vdi_d    = to_drought_positive(vdi_m,   "vdi")
pdsi_d   = to_drought_positive(pdsi_m,  "pdsi")
vhi_d    = to_drought_positive(vhi_m,   "vhi")
spi3_d   = to_drought_positive(spi3_m,  "spi")
spi6_d   = to_drought_positive(spi6_m,  "spi")
grace_d  = to_drought_positive(grace_m, "grace")
eddi3_d  = to_drought_positive(eddi3_m, "eddi")
eddi6_d  = to_drought_positive(eddi6_m, "eddi")

panel_sources = [
    ("VDI",   vdi_d),
    ("PDSI (dry=+)",  pdsi_d),
    ("VHI drought",   vhi_d),
    ("SPI-3 (dry=+)", spi3_d),
    ("SPI-6 (dry=+)", spi6_d),
    ("GRACE (dry=+)", grace_d),
    ("EDDI-3",        eddi3_d),
    ("EDDI-6",        eddi6_d),
]

def plot_bars_panels_for_biome(biome_name, biome_mask, outdir):
    """
    One figure per biome with 8 panels (variables).
    Bars are colored by standardized value using a green→yellow→red gradient.
    Shared y-limits and color normalization across panels within biome.
    """
    # 1) Build area-weighted monthly series for each variable, masked to biome
    series = []
    labels = []
    for label, da in panel_sources:
        da_b = apply_mask(da, biome_mask, thr=0.5)
        ts   = area_weighted_biome_mean(da_b)  # 1D monthly series
        series.append(ts)
        labels.append(label)

    # 2) Use common overlap so all panels share the same x-axis
    series_aligned, common_time = intersect_many(series)

    # 3) Standardize each series; collect global max abs for common limits
    series_std = [zscore(s) for s in series_aligned]
    if len(series_std) == 0 or len(common_time) == 0:
        print(f"Skipping {biome_name}: no overlapping data.")
        return

    # Clip extreme outliers for stable color/y scaling
    all_vals = np.concatenate([np.asarray(s.values, dtype=float) for s in series_std])
    finite = all_vals[np.isfinite(all_vals)]
    if finite.size == 0:
        print(f"Skipping {biome_name}: all-NaN after masking.")
        return
    # robust symmetric limit
    lim = float(np.nanpercentile(np.abs(finite), 98))
    lim = max(lim, 2.0)  # at least ±2σ
    norm = TwoSlopeNorm(vmin=-lim, vcenter=0.0, vmax=lim)
    cmap = mpl.colormaps.get("RdYlGn_r")  # green (wet/−) -> yellow -> red (dry/+)

    # 4) Make 4x2 panel figure
    fig, axes = plt.subplots(4, 2, figsize=(14, 10), sharex=True)
    fig.patch.set_facecolor("white")
    axes = axes.ravel()

    # Bar width ~ 27 days so adjacent months touch nicely
    # Convert time coords (np.datetime64) -> pandas Timestamps for matplotlib
    dates = pd.to_datetime(common_time)
    width = 27  # days

    for ax, s, lab in zip(axes, series_std, labels):
        vals = np.asarray(s.values, dtype=float)
        # Colors by value
        colors = cmap(norm(vals))
        # Bars centered at month
        ax.bar(dates, vals, width=width, align="center", color=colors, edgecolor="none")
        ax.axhline(0, color="k", linewidth=0.8, alpha=0.6)
        ax.set_ylim(-lim, lim)
        ax.set_title(lab, fontsize=11, pad=4)
        ax.grid(True, axis="y", alpha=0.25)
        # cleaner axes
        ax.tick_params(axis='x', labelrotation=0)

    # Hide any unused axes (in case fewer than 8 panels ever used)
    for j in range(len(series_std), len(axes)):
        axes[j].axis("off")

    # Shared colorbar (right)
    sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    cax = fig.add_axes([0.92, 0.15, 0.02, 0.70])
    cb = fig.colorbar(sm, cax=cax)
    cb.set_label("Standardized drought anomaly (dry +, wet −)", fontsize=10)
    cb.ax.tick_params(labelsize=9)

    fig.suptitle(f"{biome_name.capitalize()} — Monthly standardized drought (area-weighted mean)", fontsize=14)
    plt.subplots_adjust(left=0.06, right=0.90, top=0.93, bottom=0.06, hspace=0.25)

    out_png = os.path.join(outdir, f"Timeseries_panels_{biome_name}.png")
    fig.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {out_png}")

print("Making multi-panel bar figures per biome (sign-harmonized)…")
for bname, bmask in biome_masks.items():
    plot_bars_panels_for_biome(bname, bmask, paths["out"])

print("Done generating panel time-series figures.")


Making multi-panel bar figures per biome (sign-harmonized)…
Saved: C:\Drought\Maps_out\Timeseries_panels_cropland.png
Saved: C:\Drought\Maps_out\Timeseries_panels_forest.png
Saved: C:\Drought\Maps_out\Timeseries_panels_shrub_grass.png
Done generating panel time-series figures.


In [5]:
from matplotlib.colors import TwoSlopeNorm
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def area_weighted_india_mean(da):
    """
    Area-weighted (cos lat) mean over all India pixels (no biome mask).
    Uses a stencil of pixels that are ever valid over time.
    Returns 1D DataArray over 'time'.
    """
    lat = da["lat"]
    w_lat = np.cos(np.deg2rad(lat))
    W, _ = xr.broadcast(w_lat, da["lon"])
    # stencil where pixel is valid in ANY month (more robust than using first month)
    stencil = np.isfinite(da).any(dim="time")
    W = W.where(stencil, 0.0)
    return da.weighted(W).mean(("lat", "lon"), skipna=True)

def plot_bars_panels_india(outdir):
    """
    One figure with 8 panels (variables) for whole India.
    Bars colored by standardized value (green→yellow→red).
    Shared y-limits and color normalization across panels.
    """
    # 1) Build national area-weighted monthly series for each variable
    series = []
    labels = []
    for label, da in panel_sources:
        ts = area_weighted_india_mean(da)
        series.append(ts)
        labels.append(label)

    # 2) Common overlap for shared x-axis
    series_aligned, common_time = intersect_many(series)

    # 3) Standardize and set shared symmetric limits
    series_std = [zscore(s) for s in series_aligned]
    if len(series_std) == 0 or len(common_time) == 0:
        print("Skipping India figure: no overlapping data.")
        return

    all_vals = np.concatenate([np.asarray(s.values, dtype=float) for s in series_std])
    finite = all_vals[np.isfinite(all_vals)]
    if finite.size == 0:
        print("Skipping India figure: all-NaN after area averaging.")
        return
    lim = float(np.nanpercentile(np.abs(finite), 98))
    lim = max(lim, 2.0)  # at least ±2σ
    norm = TwoSlopeNorm(vmin=-lim, vcenter=0.0, vmax=lim)
    cmap = mpl.colormaps.get("RdYlGn_r")  # wet (−)→green, dry (+)→red

    # 4) Plot 4x2 panels
    fig, axes = plt.subplots(4, 2, figsize=(14, 10), sharex=True)
    fig.patch.set_facecolor("white")
    axes = axes.ravel()

    dates = pd.to_datetime(common_time)
    width = 27  # days, so bars touch nicely

    for ax, s, lab in zip(axes, series_std, labels):
        vals = np.asarray(s.values, dtype=float)
        colors = cmap(norm(vals))
        ax.bar(dates, vals, width=width, align="center", color=colors, edgecolor="none")
        ax.axhline(0, color="black", linewidth=0.8, alpha=0.6)
        ax.set_ylim(-lim, lim)
        ax.set_title(lab, fontsize=11, pad=4)
        ax.grid(True, axis="y", alpha=0.25)

    # Hide any unused axes (defensive)
    for j in range(len(series_std), len(axes)):
        axes[j].axis("off")

    # Shared colorbar
    sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    cax = fig.add_axes([0.92, 0.15, 0.02, 0.70])
    cb = fig.colorbar(sm, cax=cax)
    cb.set_label("Standardized drought anomaly (dry +, wet −)", fontsize=10)
    cb.ax.tick_params(labelsize=9)

    fig.suptitle("India — Monthly standardized drought (area-weighted mean)", fontsize=14)
    plt.subplots_adjust(left=0.06, right=0.90, top=0.93, bottom=0.06, hspace=0.25)

    out_png = os.path.join(paths["out"], "Timeseries_panels_India.png")
    fig.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {out_png}")

# Run it
print("Making multi-panel bar figure for ALL India (sign-harmonized)…")
plot_bars_panels_india(paths["out"])
print("Done.")


Making multi-panel bar figure for ALL India (sign-harmonized)…
Saved: C:\Drought\Maps_out\Timeseries_panels_India.png
Done.


In [7]:
import os
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
import geopandas as gpd

# ----------------------------
# Paths (edit 'root' if needed)
# ----------------------------
root = r"C:\Drought"
paths = {
    "VDI_dir": os.path.join(root, r"Regridding and data clipping\Final Analysis\VDI"),
    "PDSI":    os.path.join(root, r"Regridding and data clipping\PDSI", "TerraClimate_PDSI_0p05deg_INDIA.nc"),
    "VHI":     os.path.join(root, r"Regridding and data clipping\VHI", "MODIS_VCI_TCI_VHI_India_0p05deg_2001-2023.nc"),
    "SPI":     os.path.join(root, r"SPI", "SPI_CHIRPS_k_1_3_6_9_12_2000_2023_0p05deg.nc"),
    "GRACE":   os.path.join(root, r"Regridding and data clipping\GRACE", "GRACE_JPL_mascon_India_0p05deg_2003-2023.nc"),
    "EDDI":    os.path.join(root, r"EDDI", "EDDI_005deg_200012-202312.nc"),
    "LULC_dir":os.path.join(root, r"Land Cover"),
    "shp":     os.path.join(root, r"Land Cover", "India_with_jk.shp"),
    "out":     os.path.join(root, r"Maps_out"),
}
os.makedirs(paths["out"], exist_ok=True)

# ----------------------------
# Utilities
# ----------------------------
def open_nc(fp):
    """Open NetCDF, standardize lat/lon names & sort ascending."""
    ds = xr.open_dataset(fp)
    rename = {}
    if "latitude" in ds: rename["latitude"] = "lat"
    if "longitude" in ds: rename["longitude"] = "lon"
    if "y" in ds and "lat" not in ds: rename["y"] = "lat"
    if "x" in ds and "lon" not in ds: rename["x"] = "lon"
    if rename: ds = ds.rename(rename)
    if "lat" in ds.coords: ds = ds.sortby("lat")
    if "lon" in ds.coords: ds = ds.sortby("lon")
    return ds

def first_var(ds, candidates):
    """Pick first present var from candidates (case-insensitive) else first time/lat/lon var."""
    cl = [c.lower() for c in candidates]
    for v in ds.data_vars:
        if v.lower() in cl: return v
    for v in candidates:
        if v in ds.data_vars: return v
    for v in ds.data_vars:
        dims = ds[v].dims
        if "time" in dims and (("lat" in dims) or ("y" in dims)) and (("lon" in dims) or ("x" in dims)):
            return v
    raise ValueError(f"Could not find expected variable among: {list(ds.data_vars)}")

def ensure_datetime_monthly(da):
    """Ensure monthly means at month-start timestamps."""
    if "time" not in da.dims:
        raise ValueError("DataArray must have 'time' dimension.")
    if not np.issubdtype(da["time"].dtype, np.datetime64):
        da = da.assign_coords(time=pd.to_datetime(da["time"].values))
    da = da.sortby("time")
    if not np.issubdtype(da.dtype, np.number):
        da = da.astype("float64")
    da_m = da.resample(time="MS").mean(skipna=True)
    # drop months that are all-NaN everywhere (cleaner)
    if "time" in da_m.coords:
        all_nan = ~np.isfinite(da_m).any(dim=[d for d in da_m.dims if d != "time"])
        if all_nan.any():
            keep = (~all_nan).values
            if keep.any():
                da_m = da_m.sel(time=da_m["time"].values[keep])
    return da_m

def interp_to_ref_grid(da, ref_lat, ref_lon, method="nearest"):
    """Interpolate onto reference (lat,lon) grid if needed."""
    if "lat" not in da.coords and "y" in da.coords: da = da.rename({"y":"lat"})
    if "lon" not in da.coords and "x" in da.coords: da = da.rename({"x":"lon"})
    if "lat" in da.coords: da = da.sortby("lat")
    if "lon" in da.coords: da = da.sortby("lon")
    same_lat = ("lat" in da.coords) and np.array_equal(da["lat"].values, ref_lat.values)
    same_lon = ("lon" in da.coords) and np.array_equal(da["lon"].values, ref_lon.values)
    if same_lat and same_lon:
        return da
    return da.interp(lat=ref_lat, lon=ref_lon, method=method)

def apply_mask(da, mask, thr=0.5):
    """Mask with LULC fraction >= thr (interpolate mask if needed)."""
    if "lat" in mask.coords: mask = mask.sortby("lat")
    if "lon" in mask.coords: mask = mask.sortby("lon")
    if "time" in mask.dims: mask = mask.mean("time")
    need_interp = (
        mask.sizes.get("lat", -1) != da.sizes.get("lat", -1) or
        mask.sizes.get("lon", -1) != da.sizes.get("lon", -1) or
        not np.array_equal(mask.get_index("lat"), da.get_index("lat")) or
        not np.array_equal(mask.get_index("lon"), da.get_index("lon"))
    )
    if need_interp:
        mask = mask.interp(lat=da["lat"], lon=da["lon"], method="nearest")
    mask_bool = xr.where(mask >= thr, 1.0, np.nan)
    return da * mask_bool

def to_drought_positive(da, kind):
    """Convert index so that larger = drier."""
    if kind == "vdi":   return da
    if kind == "eddi":  return da
    if kind == "pdsi":  return -da
    if kind == "spi":   return -da
    if kind == "grace": return -da
    if kind == "vhi":
        vmin = float(da.min(skipna=True))
        vmax = float(da.max(skipna=True))
        if 0.0 <= vmin <= 100.0 and 0.0 <= vmax <= 100.0:
            return 100.0 - da
        else:
            return -da
    raise ValueError(f"Unknown kind={kind}")

def stack_pairs(vdi_da, other_da, mask=None):
    """
    Stack all (time, lat, lon) pairs into 1D arrays (x=VDI, y=Other),
    restricted to overlapping times and optional biome mask (>=0.5).
    """
    # time overlap
    ta = pd.Index(pd.to_datetime(vdi_da["time"].values))
    tb = pd.Index(pd.to_datetime(other_da["time"].values))
    common = np.array(sorted(ta.intersection(tb)))
    a = vdi_da.sel(time=common)
    b = other_da.sel(time=common)

    # optional mask
    if mask is not None:
        a = apply_mask(a, mask, thr=0.5)
        b = apply_mask(b, mask, thr=0.5)

    # flatten
    xa = a.values.reshape(-1)
    yb = b.values.reshape(-1)

    # finite only
    m = np.isfinite(xa) & np.isfinite(yb)
    return xa[m], yb[m]

def robust_limits(x, y):
    """Symmetric robust axis limits using 2–98th percentiles of both arrays."""
    both = np.concatenate([x, y])
    finite = both[np.isfinite(both)]
    if finite.size < 10:
        return -1, 1
    lo = np.nanpercentile(finite, 2)
    hi = np.nanpercentile(finite, 98)
    lim = max(abs(lo), abs(hi))
    lim = max(lim, 1.0)
    return -lim, lim

def panel_density(ax, x, y, title, bins=120, cmap="rainbow", extent=None, vmax=None):
    """
    Draw 2D histogram (density scatter) with y=x line.
    extent: (xmin, xmax, ymin, ymax). vmax: shared color max for normalization.
    """
    if extent is None:
        xmin, xmax = robust_limits(x, y)
        ymin, ymax = xmin, xmax
    else:
        xmin, xmax, ymin, ymax = extent

    # 2D histogram
    H, xedges, yedges = np.histogram2d(x, y, bins=bins,
                                       range=[[xmin, xmax], [ymin, ymax]])
    # transpose so x along axis0, y along axis1 for pcolormesh
    H = H.T

    # plot
    X, Y = np.meshgrid(xedges, yedges)
    norm = Normalize(vmin=0, vmax=vmax if vmax is not None else H.max())
    im = ax.pcolormesh(X, Y, H, cmap=cmap, norm=norm, shading="auto")

    # y=x line
    ax.plot([xmin, xmax], [xmin, xmax], color="black", linewidth=1.2)

    # axes & title
    ax.set_xlim([xmin, xmax]); ax.set_ylim([ymin, ymax])
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("VDI (dry +)")
    ax.set_ylabel(title.replace("VDI vs ", "") + " (dry +)")
    ax.grid(True, alpha=0.15)

    return im, H.max(), (xmin, xmax, ymin, ymax)

# ----------------------------
# Load masks & data
# ----------------------------
print("Loading masks & datasets...")

# Biome masks
mask_files = {
    "cropland":    os.path.join(paths["LULC_dir"], "lulc_cropland_frac_0p05deg.nc"),
    "forest":      os.path.join(paths["LULC_dir"], "lulc_forest_frac_0p05deg.nc"),
    "shrub_grass": os.path.join(paths["LULC_dir"], "lulc_shrub_grass_frac_0p05deg.nc"),
}
biome_masks = {}
for name, fp in mask_files.items():
    ds = open_nc(fp)
    v = first_var(ds, ["fraction","frac","cropland","forest","shrub_grass","lulc"])
    m = ds[v]
    if "y" in m.dims and "lat" not in m.dims: m = m.rename({"y":"lat"})
    if "x" in m.dims and "lon" not in m.dims: m = m.rename({"x":"lon"})
    if "time" in m.dims: m = m.mean("time")
    biome_masks[name] = m

# India shapefile is not needed for these plots (no maps), but path kept in 'paths'.

# Load gridded datasets
def pick_vdi_file():
    cands = [f for f in os.listdir(paths["VDI_dir"]) if f.lower().endswith(".nc")]
    pref  = [f for f in cands if "vdi" in f.lower() and ("all" in f.lower() or "india" in f.lower())]
    return os.path.join(paths["VDI_dir"], pref[0] if pref else cands[0])

vdi_ds = open_nc(pick_vdi_file()); vdi = vdi_ds[first_var(vdi_ds, ["VDI","vdi"])]
pdsi_ds = open_nc(paths["PDSI"]); pdsi = pdsi_ds[first_var(pdsi_ds, ["PDSI","pdsi"])]
vhi_ds  = open_nc(paths["VHI"]);  vhi  = vhi_ds[first_var(vhi_ds,  ["VHI","vhi"])]
spi_ds  = open_nc(paths["SPI"])
spi3    = spi_ds[first_var(spi_ds, ["SPI3","SPI_3","spi3","spi_3"])]
spi6    = spi_ds[first_var(spi_ds, ["SPI6","SPI_6","spi6","spi_6"])]
grace_ds= open_nc(paths["GRACE"]); grace = grace_ds[first_var(grace_ds, ["TWS","tws","tws_anom","GAB_msc_Lmax180","grace","mascon"])]
eddi_ds = open_nc(paths["EDDI"])
eddi3   = eddi_ds[first_var(eddi_ds, ["EDDI_03","EDDI3","eddi_03","EDDI_3","eddi3"])]
eddi6   = eddi_ds[first_var(eddi_ds, ["EDDI_06","EDDI6","eddi_06","EDDI_6","eddi6"])]

# Monthly + align to VDI grid
vdi_m   = ensure_datetime_monthly(vdi)
ref_lat, ref_lon = vdi_m["lat"], vdi_m["lon"]
pdsi_m  = interp_to_ref_grid(ensure_datetime_monthly(pdsi),  ref_lat, ref_lon)
vhi_m   = interp_to_ref_grid(ensure_datetime_monthly(vhi),   ref_lat, ref_lon)
spi3_m  = interp_to_ref_grid(ensure_datetime_monthly(spi3),  ref_lat, ref_lon)
spi6_m  = interp_to_ref_grid(ensure_datetime_monthly(spi6),  ref_lat, ref_lon)
grace_m = interp_to_ref_grid(ensure_datetime_monthly(grace), ref_lat, ref_lon)
eddi3_m = interp_to_ref_grid(ensure_datetime_monthly(eddi3), ref_lat, ref_lon)
eddi6_m = interp_to_ref_grid(ensure_datetime_monthly(eddi6), ref_lat, ref_lon)

# Sign harmonization (higher = drier)
vdi_d   = to_drought_positive(vdi_m,   "vdi")
pdsi_d  = to_drought_positive(pdsi_m,  "pdsi")
vhi_d   = to_drought_positive(vhi_m,   "vhi")
spi3_d  = to_drought_positive(spi3_m,  "spi")
spi6_d  = to_drought_positive(spi6_m,  "spi")
grace_d = to_drought_positive(grace_m, "grace")
eddi3_d = to_drought_positive(eddi3_m, "eddi")
eddi6_d = to_drought_positive(eddi6_m, "eddi")

# Variables to compare with VDI
targets = [
    ("VDI vs PDSI (dry=+)",  pdsi_d,  "pdsi"),
    ("VDI vs VHI drought",    vhi_d,   "vhi"),
    ("VDI vs SPI-3 (dry=+)",  spi3_d,  "spi3"),
    ("VDI vs SPI-6 (dry=+)",  spi6_d,  "spi6"),
    ("VDI vs EDDI-3",         eddi3_d, "eddi3"),
    ("VDI vs EDDI-6",         eddi6_d, "eddi6"),
]

# ----------------------------
# Make figures
# ----------------------------
def make_density_figure(region_name, mask=None, outfile="out.png", bins=120, cmap="rainbow"):
    """
    Build a 2x3 panel density figure for the region (mask=None => India).
    One shared colorbar on the right.
    """
    # Stack pairs for each variable to determine global extents and max density
    pairs = []
    maxcounts = []
    extents = []
    for title, other_da, _ in targets:
        x, y = stack_pairs(vdi_d, other_da, mask=mask)
        xmin, xmax = robust_limits(x, y)
        extents.append((xmin, xmax, xmin, xmax))
        # quick max density estimate for shared colorbar
        H, _, _ = np.histogram2d(x, y, bins=bins, range=[[xmin, xmax], [xmin, xmax]])
        maxcounts.append(H.max())
        pairs.append((title, x, y))

    shared_vmax = float(np.nanmax(maxcounts))
    # Panel plot
    fig, axes = plt.subplots(2, 3, figsize=(13, 7))
    fig.patch.set_facecolor("white")
    axes = axes.ravel()

    mappable = None
    for ax, (title, x, y), extent in zip(axes, pairs, extents):
        im, _, _ = panel_density(ax, x, y, title, bins=bins, cmap=cmap, extent=extent, vmax=shared_vmax)
        mappable = im

    # Shared colorbar (normalized counts)
    cax = fig.add_axes([0.92, 0.15, 0.02, 0.70])
    cb = fig.colorbar(mappable, cax=cax)
    cb.set_label("Point density (normalized counts)", fontsize=10)
    cb.ax.tick_params(labelsize=9)

    fig.suptitle(f"{region_name} — VDI vs other drought metrics (higher = drier)", fontsize=14)
    plt.subplots_adjust(left=0.06, right=0.90, top=0.92, bottom=0.08, wspace=0.20, hspace=0.25)
    fig.savefig(outfile, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {outfile}")

print("Generating density figures (VDI vs others)…")

# India (no mask)
make_density_figure("India",
                    mask=None,
                    outfile=os.path.join(paths["out"], "Density_VDI_vs_others_India.png"))

# Biomes
for bname, bmask in biome_masks.items():
    make_density_figure(bname.capitalize(),
                        mask=bmask,
                        outfile=os.path.join(paths["out"], f"Density_VDI_vs_others_{bname}.png"))

print("All done.")


Loading masks & datasets...
Generating density figures (VDI vs others)…
Saved: C:\Drought\Maps_out\Density_VDI_vs_others_India.png
Saved: C:\Drought\Maps_out\Density_VDI_vs_others_cropland.png
Saved: C:\Drought\Maps_out\Density_VDI_vs_others_forest.png
Saved: C:\Drought\Maps_out\Density_VDI_vs_others_shrub_grass.png
All done.
